# Option Pricing Interactive Dashboard

This dashboard integrates Black-Scholes, Binomial Tree, and Monte Carlo models for real-time option analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, interactive_output, HBox, VBox
from IPython.display import display, clear_output

# Import models
from black_scholes import BlackScholesModel
from binomial import BinomialTreeModel
from monte_carlo import MonteCarloModel

bs_model = BlackScholesModel()
bt_model = BinomialTreeModel()
mc_model = MonteCarloModel()

In [ ]:
def update_dashboard(S, K, T, r, sigma, option_type, american):
    # Calculate Prices
    bs_price, greeks = bs_model.price_option(S, K, T, r, sigma, option_type)
    bt_price, stock_tree, option_tree = bt_model.price_option(S, K, T, r, sigma, 20, option_type, american)
    mc_price, ci = mc_model.price_option(S, K, T, r, sigma, num_simulations=5000, option_type=option_type)
    
    # Create Layout
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Black-Scholes Price vs Stock Price
    S_range = np.linspace(S*0.5, S*1.5, 50)
    bs_prices = [bs_model.price_option(s, K, T, r, sigma, option_type)[0] for s in S_range]
    payoff = [max(0, s-K) if option_type == 'call' else max(0, K-s) for s in S_range]
    
    axes[0].plot(S_range, bs_prices, label='BS Price', color='blue', lw=2)
    axes[0].plot(S_range, payoff, 'r--', label='Intrinsic Value', alpha=0.7)
    axes[0].axvline(S, color='green', linestyle=':', label=f'Current S=${S}')
    axes[0].set_title(f'Black-Scholes {option_type.capitalize()} Price Surface')
    axes[0].set_xlabel('Stock Price ($)')
    axes[0].set_ylabel('Option Price ($)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Greeks Radar/Bar Chart (Simple visualization)
    greek_names = list(greeks.keys())
    greek_values = [greeks[g] for g in greek_names]
    # Normalize for display if needed, but here we just plot
    axes[1].bar(greek_names, greek_values, color='skyblue')
    axes[1].set_title('Option Greeks')
    axes[1].grid(True, axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print Summary Table
    print(f"{'MODEL':<20} | {'PRICE':<10}")
    print("-" * 35)
    print(f"{'Black-Scholes':<20} | ${bs_price:.4f}")
    print(f"{'Binomial Tree (N=20)':<20} | ${bt_price:.4f}")
    print(f"{'Monte Carlo (N=5000)':<20} | ${mc_price:.4f}")
    print(f"\nMC Confidence Interval: (${ci[0]:.2f}, ${ci[1]:.2f})")
    
    print("\nGREEKS (Black-Scholes):")
    for g, v in greeks.items():
        print(f"  {g.capitalize():<6}: {v:.6f}")

In [ ]:
# Define Widgets
S_slider = widgets.FloatSlider(value=100, min=10, max=200, step=1.0, description='Stock (S)')
K_slider = widgets.FloatSlider(value=100, min=10, max=200, step=1.0, description='Strike (K)')
T_slider = widgets.FloatSlider(value=1.0, min=0.01, max=5.0, step=0.1, description='Time (T)')
r_slider = widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.01, description='Rate (r)')
sigma_slider = widgets.FloatSlider(value=0.2, min=0.01, max=1.0, step=0.01, description='Vol (σ)')
type_dropdown = widgets.Dropdown(options=['call', 'put'], value='call', description='Type')
american_check = widgets.Checkbox(value=False, description='American')

# Arrange Widgets
controls = VBox([S_slider, K_slider, T_slider, r_slider, sigma_slider, type_dropdown, american_check])
out = interactive_output(update_dashboard, {
    'S': S_slider, 'K': K_slider, 'T': T_slider, 'r': r_slider, 
    'sigma': sigma_slider, 'option_type': type_dropdown, 'american': american_check
})

display(HBox([controls, out]))